## Pod phases, conditions & container states

A pod's `status` gives three overlapping views, each answering a different question.

### `status.phase` — the headline

One mutually-exclusive word:

| Phase | Meaning |
|---|---|
| `Pending` | Accepted, but a container hasn't started — scheduling or image pull. |
| `Running` | Scheduled, all containers created, at least one up. |
| `Succeeded` | All containers exited `0`, won't restart. |
| `Failed` | All terminated, at least one failed. |
| `Unknown` | The node stopped reporting in. |

### `status.conditions` — the deeper view

Each condition is `True` or `False`, and they flip **in order**:

- **`PodScheduled`** — a node was picked.
- **`Initialized`** — all init containers finished.
- **`ContainersReady`** — every readiness probe passes.
- **`Ready`** — the pod can receive traffic.

A pod stuck on a failing image pull shows `PodScheduled: True` but `Initialized: False`, `Ready: False` — and the `message` field says why.

### `status.containerStatuses` — per container

Each container is in one of three states: **`waiting`** (`reason`: `ContainerCreating`, `ImagePullBackOff`, `CrashLoopBackOff`), **`running`** (`startedAt`), or **`terminated`** (`exitCode`, `finishedAt`). Plus **`restartCount`** — a growing count is a red flag even while the pod reads "Running."

### The troubleshooting flow

```bash
kubectl get pods                  # phase + restart count at a glance
kubectl describe pod <name>       # conditions, events, container reasons
kubectl logs <name> [--previous]  # what the app said before it died
```

`describe` is the workhorse — its `Events:` section is the single richest "what just went wrong." On our map you're reading the state the **kubelet** reports up about each **Pod** — the actual half of the spec/status pair.